In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.5/175.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 

In [33]:
!pip install faiss

ERROR: Could not find a version that satisfies the requirement faiss (from versions: none)
ERROR: No matching distribution found for faiss


In [62]:
import torch
import numpy as np
import json
from tqdm import tqdm
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
from collections import Counter
import os

In [54]:
with open("sample_data/course_goals.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [55]:
course_goals_texts = [example["course_goals"] for example in data if "course_goals" in example]

In [56]:
max_seq_length = 1024
dtype = torch.float16
load_in_4bit = True

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Meta-Llama-3.1-8b",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.1.8: Fast Llama patching. Transformers: 4.47.1.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

In [64]:
def get_embedding(text, model, tokenizer, max_length=512):
    """Tworzy embedding dla podanego tekstu przy użyciu modelu."""
    input_ids = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length).input_ids.to(model.device)
    with torch.no_grad():
        output = model(input_ids, output_hidden_states=True)
    embedding = output.hidden_states[-1].mean(dim=1).cpu().numpy()
    return embedding

In [59]:
embedding_file = "course_goals_embeddings.npy"

In [63]:
if os.path.exists(embedding_file):
    print("🔄 Ładowanie zapisanych embeddingów...")
    course_goals_embeddings = np.load(embedding_file)
else:
    print("🔄 Generowanie embeddingów dla kursów...")

    course_goals_embeddings = np.vstack(
        [get_embedding(text, model, tokenizer) for text in tqdm(course_goals_texts, desc="Postęp", unit="kurs")]
    )

    course_goals_embeddings = normalize(course_goals_embeddings)

    np.save(embedding_file, course_goals_embeddings)
    print(f"Embeddingi zapisane do pliku: {embedding_file}")

🔄 Generowanie embeddingów dla kursów...



Postęp:   0%|          | 0/1658 [00:00<?, ?kurs/s]

TypeError: get_embedding() takes 1 positional argument but 3 were given

In [49]:
user_query = input("Podaj opis kursu, który Cię interesuje: ")

user_embedding = get_embedding(user_query)

similarities = cosine_similarity(user_embedding, course_goals_embeddings)[0]

best_match_idx = np.argmax(similarities)
best_match_text = course_goals_texts[best_match_idx]

print("\n Najbardziej dopasowany kurs:")
print(best_match_text)

🔄 Generowanie embeddingów dla kursów...


Postęp:   0%|          | 2/631 [00:06<32:39,  3.12s/kurs]

KeyboardInterrupt: 